# 04 · BW + PLI with EMD and adaptive filter (paper Figs. 6, 7, 8)

Proposed method (Fig. 3, Eq. 1): PLI filter on IMF1, BW filter on IMF5–8,
Y = IMF1′ + IMF2 + IMF3 + IMF4 + IMF′58 + r.

* **Fig. 6:** time and frequency domain example (record 119, SNRin = 10 dB)
* **Fig. 7:** LMS vs NLMS vs RLS at SNRin = −5, 0, 5, 7, 10 dB (records of Fig. 8). NLMS and RLS use MATLAB defaults (NLMS step 0.1; RLS forgetting factor 1, initial inverse covariance I), with the same number of taps as LMS.
* **Fig. 8:** SNR output on records 119, 202, 213, 217, 234 at SNRin = 10 dB, next to the values the paper reports for itself and for articles [28], [21], [22] (read off the paper's Fig. 8, ±0.3 dB).

In [ ]:
from pathlib import Path
import sys, json

project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

RESULTS = project_root / "results"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
METHOD_COLORS = {"PEAF": "#1f3b73", "SEAF": "#9aa83a", "PDAF": "#d62728", "SDAF": "#3cb9a0"}


In [ ]:
bw = pd.read_csv(RESULTS / "bw_pli.csv")
reported = pd.read_csv(RESULTS / "paper_fig8_reported.csv")
print("LMS setting:", json.loads((RESULTS / "lms_setting.json").read_text()))

## Fig. 6: example (record 119, SNRin = 10 dB, first 5000 samples as in the paper)

In [ ]:
ex = np.load(RESULTS / "fig6_example.npz"); fs = int(ex["fs"])
sigs = [("original signal", ex["clean"]), ("signal + noise", ex["noisy"]), ("EMD denoised signal", ex["EMD+LMS"])]
fig, axes = plt.subplots(3, 2, figsize=(16, 8), gridspec_kw={"width_ratios": [1.6, 1]})
for (name, s), (ax_t, ax_f) in zip(sigs, axes):
    ax_t.plot(s[:5000], lw=0.6); ax_t.set_title(name, fontsize=10); ax_t.set_ylabel("mV")
    f = np.fft.rfftfreq(len(s), 1 / fs)
    ax_f.plot(f, np.abs(np.fft.rfft(s - s.mean())) / len(s), lw=0.5)
    ax_f.set_xlim(0, 180); ax_f.set_title(name + " (spectrum, whole record)", fontsize=10)
axes[-1, 0].set_xlabel("sample"); axes[-1, 1].set_xlabel("f (Hz)")
plt.tight_layout(); plt.show()

### Zoom: convergence at the start and a later 3 s window

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
for ax, start in zip(axes, [0, 600 * fs]):
    w = slice(start, start + 3 * fs); t = np.arange(w.start, w.stop) / fs
    ax.plot(t, ex["noisy"][w], lw=0.6, color="0.7", label="signal + noise")
    ax.plot(t, ex["clean"][w], lw=1.2, color="k", label="original")
    ax.plot(t, ex["EMD+LMS"][w], lw=0.9, color="tab:orange", label="EMD denoised")
    ax.set_xlabel("time (s)"); ax.set_ylabel("mV"); ax.legend(fontsize=8)
axes[0].set_title("first 3 s (adaptive filter converging)"); axes[1].set_title("t = 600 s")
plt.tight_layout(); plt.show()

## Fig. 7: LMS vs NLMS vs RLS (mean over the 5 records)

In [ ]:
m = bw.groupby(["SNRin_dB", "filter"])[["SNRout_dB", "RMSE"]].mean().unstack()
snrs = sorted(bw.SNRin_dB.unique()); x = np.arange(len(snrs))
colors = {"LMS": "#9acd32", "RLS": "#e00000", "NLMS": "#3cb4e6"}
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
for ax, metric, ylabel in [(axes[0], "SNRout_dB", "SNR output (dB)"), (axes[1], "RMSE", "RMSE (mV)")]:
    for k, f in enumerate(["LMS", "RLS", "NLMS"]):
        ax.bar(x + (k - 1) * 0.25, m[metric][f].values, 0.25, label=f + "+EMD", color=colors[f])
    ax.set_xticks(x, [f"SNRin = {s} dB" for s in snrs]); ax.set_ylabel(ylabel); ax.legend()
axes[0].set_title("SNR output under different SNRinput"); axes[1].set_title("RMSE under different SNRinput")
plt.tight_layout(); plt.show()
m.round(3)

In [ ]:
best = m["SNRout_dB"].idxmax(axis=1)
print("best filter per SNRin:", best.to_dict())
print("paper: LMS is best and most stable -> holds here:", bool((best == "LMS").all()))

## Fig. 8: SNR output at SNRin = 10 dB, ours vs values reported in the paper

In [ ]:
ours = bw[(bw.SNRin_dB == 10) & (bw["filter"] == "LMS")].set_index("record")
rec = sorted(ours.index)
style = {"Proposed LMS+EMD (paper)": ("#1f3b73", "--"), "Article [28]": ("#7b3fb0", "-"),
         "Article [21]": ("#e00000", "-"), "Article [22]": ("#3cb9a0", "-")}
fig, ax = plt.subplots(figsize=(11, 4.8))
for name, g in reported.groupby("method", sort=False):
    c, ls = style[name]
    ax.plot([str(r) for r in g.record], g.SNRout_dB, ls, marker="o", color=c, label=name + " (reported)")
ax.plot([str(r) for r in rec], ours.loc[rec, "SNRout_dB"], "-", marker="s", lw=2.5, color="tab:orange",
        label="Proposed LMS+EMD (this reproduction)")
ax.set_xlabel("Signal number"); ax.set_ylabel("SNR (dB)"); ax.set_ylim(0, 35)
ax.set_title("Comparison of the SNR of different denoising algorithm (SNRin = 10 dB)"); ax.legend(fontsize=8)
plt.show()
table = reported.pivot(index="record", columns="method", values="SNRout_dB")
table["This reproduction"] = ours["SNRout_dB"]; table["This reproduction RMSE"] = ours["RMSE"]
table.round(2)